In [1]:
from pathlib import Path
import math
import gc
import numpy as np
import pandas as pd
 
csv_path = Path("masterDataOptimal_v220230323015634.csv")
state_path = Path("period_transition_custom_params_results/period_state_assignments.csv")
 
output_dir = Path("cox_vs_rsf_2012_cluster1_results")
output_dir.mkdir(parents=True, exist_ok=True)
 
baseline_date = pd.Timestamp("2012-03-01")
followup_end_date = pd.Timestamp("2022-03-01")
 
source_period = "before_2012_03"
source_cluster = "Cluster 1"
 
print("CSV:", csv_path)
print("State file:", state_path)
print("Output dir:", output_dir)

CSV: masterDataOptimal_v220230323015634.csv
State file: period_transition_custom_params_results/period_state_assignments.csv
Output dir: cox_vs_rsf_2012_cluster1_results


In [2]:
# get_ipython().run_line_magic('conda', 'install -c conda-forge scikit-survival -y')
 
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
 
print("scikit-survival ready")

scikit-survival ready


In [3]:
def safe_to_csv(df, path, **kwargs):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if "index" not in kwargs:
        kwargs["index"] = False
    if "encoding" not in kwargs:
        kwargs["encoding"] = "utf-8-sig"
    try:
        df.to_csv(path, **kwargs)
        return path
    except PermissionError:
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        fallback_path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")
        df.to_csv(fallback_path, **kwargs)
        print(f"Permission denied for {path}. Saved timestamped copy instead: {fallback_path}")
        return fallback_path

In [4]:
# In[4]:select 2012.03 Cluster 1 patients
 
state_df = pd.read_csv(state_path)
 
cluster_rows = state_df.loc[
    state_df[source_period] == source_cluster,
    "original_row_index",
].astype(int).to_numpy()
 
cluster_row_set = set(cluster_rows.tolist())
print(f"Selected {source_period} {source_cluster} participants: {len(cluster_rows):,}")

Selected before_2012_03 Cluster 1 participants: 14,406


In [5]:
 
all_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
 
b_cols = [c for c in all_cols if c.startswith("B_MEDI:")]
disease_pairs = []
for b_col in b_cols:
    feature_name = b_col.replace("B_MEDI:", "", 1)
    bd_col = "BD_MEDI:" + feature_name
    if bd_col in all_cols:
        disease_pairs.append((b_col, bd_col, feature_name))
 
print("Paired disease/date features:", len(disease_pairs))
 

Paired disease/date features: 73


In [6]:
 
usecols = ["DEATH_DATE", "TRANSFER_DATE", "YEAR_OF_BIRTH", "SEX"]
usecols += [col for pair in disease_pairs for col in pair[:2]]
 
def skip_non_cluster_rows(line_no):
    if line_no == 0:
        return False
    original_row_index = line_no - 1
    return original_row_index not in cluster_row_set
 
cluster_df = pd.read_csv(
    csv_path,
    usecols=usecols,
    skiprows=skip_non_cluster_rows,
    low_memory=False,
)
print("Loaded shape:", cluster_df.shape)
 

Loaded shape: (14406, 150)


In [7]:
#（time / event）
 
death_date = pd.to_datetime(cluster_df["DEATH_DATE"], errors="coerce")
transfer_date = pd.to_datetime(cluster_df["TRANSFER_DATE"], errors="coerce")
 
censor_date = pd.Series(followup_end_date, index=cluster_df.index)
censor_date = censor_date.mask(
    transfer_date.notna() & (transfer_date < censor_date),
    transfer_date,
)
 
event = death_date.notna() & (death_date >= baseline_date) & (death_date <= censor_date)
observed_end = censor_date.mask(event, death_date)
duration_days = (observed_end - baseline_date).dt.days.astype(float)
 
print("N:", len(cluster_df), "Deaths:", int(event.sum()))
 
 

N: 14406 Deaths: 6466


In [8]:
#Construct a baseline 0/1 disease feature matrix
# Diagnosis date <= 2012-03-01 Only when the disease code is not empty can it be considered that the patient was already ill at baseline
 
baseline_feature_df = pd.DataFrame(index=cluster_df.index)
 
for b_col, bd_col, feature_name in disease_pairs:
    has_disease_code = cluster_df[b_col].notna().to_numpy()
    diagnosis_date = pd.to_datetime(cluster_df[bd_col], errors="coerce")
    baseline_feature_df[feature_name] = (
        has_disease_code
        & diagnosis_date.notna().to_numpy()
        & (diagnosis_date <= baseline_date).to_numpy()
    ).astype(int)
 
baseline_feature_df["baseline_disease_count"] = baseline_feature_df.sum(axis=1)
print("Baseline feature matrix shape:", baseline_feature_df.shape)
 

Baseline feature matrix shape: (14406, 74)


In [9]:
# Construct covariates of age and gender
 
adjusted_covariate_df = pd.DataFrame(index=cluster_df.index)
 
year_of_birth_parsed = pd.to_datetime(cluster_df["YEAR_OF_BIRTH"], errors="coerce")
year_of_birth_numeric = pd.to_numeric(cluster_df["YEAR_OF_BIRTH"], errors="coerce")
year_of_birth = year_of_birth_parsed.dt.year.fillna(year_of_birth_numeric)
 
adjusted_covariate_df["age_2012"] = 2012 - year_of_birth
 
sex_raw = cluster_df["SEX"].astype("string").str.strip().fillna("Unknown")
sex_counts = sex_raw.value_counts(dropna=False)
sex_reference = sex_counts.index[0]
print("Sex reference group:", sex_reference)
print(sex_counts)
 
for sex_value in sex_counts.index[1:]:
    safe_name = str(sex_value).replace(" ", "_").replace("/", "_")
    adjusted_covariate_df[f"sex_{safe_name}"] = (sex_raw == sex_value).astype(int)
 
adjusted_model_mask = (
    adjusted_covariate_df["age_2012"].notna()
    & duration_days.notna()
    & (duration_days > 0)
)
print("Rows available for adjusted models:", int(adjusted_model_mask.sum()))

Sex reference group: M
SEX
M    7803
F    6603
Name: count, dtype: Int64
Rows available for adjusted models: 14406


In [10]:
# Core function for multivariable Cox regression using Newton-Raphson optimization of the partial likelihood
 
def _normal_two_sided_p(z):
    return math.erfc(abs(float(z)) / math.sqrt(2.0))
 
 
def fit_cox_model(time, event, X, feature_names, max_iter=60, tol=1e-7, ridge=1e-6):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=bool)
    X = np.asarray(X, dtype=float)
 
    mask = np.isfinite(time) & (time > 0) & np.isfinite(X).all(axis=1)
    time = time[mask]
    event = event[mask]
    X = X[mask]
 
    n, p = X.shape
    n_event = int(event.sum())
    if n <= p + 5 or n_event <= p + 5:
        return None
 
    X_work = X.copy()
    scale_info = []
    for j in range(p):
        values = X_work[:, j]
        unique_values = np.unique(values[~np.isnan(values)])
        is_binary = len(unique_values) <= 2 and set(unique_values).issubset({0.0, 1.0})
        if is_binary:
            scale_info.append((0.0, 1.0, True))
        else:
            mean = values.mean()
            sd = values.std(ddof=0)
            if sd <= 0:
                sd = 1.0
            X_work[:, j] = (values - mean) / sd
            scale_info.append((mean, sd, False))
 
    order = np.argsort(time)
    time = time[order]
    event = event[order]
    X_work = X_work[order]
    event_times = np.unique(time[event])
 
    beta = np.zeros(p, dtype=float)
    hessian = None
 
    for _ in range(max_iter):
        eta = np.clip(X_work @ beta, -50, 50)
        exp_eta = np.exp(eta)
        grad = np.zeros(p, dtype=float)
        info = np.zeros((p, p), dtype=float)
 
        for t in event_times:
            event_mask = (time == t) & event
            risk_mask = time >= t
            d = int(event_mask.sum())
            if d == 0:
                continue
 
            X_risk = X_work[risk_mask]
            w = exp_eta[risk_mask]
            s0 = w.sum()
            if s0 <= 0:
                continue
            s1 = (X_risk * w[:, None]).sum(axis=0)
            weighted_x = X_risk * w[:, None]
            s2 = weighted_x.T @ X_risk
 
            mean_x = s1 / s0
            grad += X_work[event_mask].sum(axis=0) - d * mean_x
            info += d * (s2 / s0 - np.outer(mean_x, mean_x))
 
        info_ridge = info + np.eye(p) * ridge
        try:
            step = np.linalg.solve(info_ridge, grad)
        except np.linalg.LinAlgError:
            step = np.linalg.pinv(info_ridge) @ grad
        beta += step
        hessian = info
        if np.max(np.abs(step)) < tol:
            break
 
    info_ridge = hessian + np.eye(p) * ridge
    try:
        cov_scaled = np.linalg.inv(info_ridge)
    except np.linalg.LinAlgError:
        cov_scaled = np.linalg.pinv(info_ridge)
 
    rows = []
    for j, name in enumerate(feature_names):
        mean, sd, is_binary = scale_info[j]
        if is_binary:
            coef = beta[j]
            se = math.sqrt(max(cov_scaled[j, j], 0))
        else:
            coef = beta[j] / sd
            se = math.sqrt(max(cov_scaled[j, j], 0)) / sd
 
        z = coef / se if se > 0 else np.nan
        p_value = _normal_two_sided_p(z) if np.isfinite(z) else np.nan
        rows.append({
            "feature": name,
            "coef": coef,
            "hr": math.exp(coef),
            "ci_low": math.exp(coef - 1.96 * se),
            "ci_high": math.exp(coef + 1.96 * se),
            "p_value": p_value,
            "se": se,
            "n": n,
            "n_event": n_event,
        })
 
    result_df = pd.DataFrame(rows)
    return result_df
 

In [11]:
#  C-index、train/test repeated k-fold 
 
class _FenwickTree:
    """Binary Indexed Tree (Fenwick tree) for efficient computation of Harrell's C-index."""
 
    def __init__(self, size):
        self.size = int(size)
        self.tree = np.zeros(self.size + 1, dtype=np.int64)
 
    def add(self, index, value):
        index = int(index)
        while index <= self.size:
            self.tree[index] += value
            index += index & -index
 
    def prefix_sum(self, index):
        index = int(index)
        total = 0
        while index > 0:
            total += self.tree[index]
            index -= index & -index
        return total
 
 
def concordance_index_censored(time, event, risk_score):
    """Harrell's C-index for right-censored survival data with an O(n log n) implementation."""
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=bool)
    risk_score = np.asarray(risk_score, dtype=float)
 
    mask = np.isfinite(time) & np.isfinite(risk_score) & (time > 0)
    time = time[mask]
    event = event[mask]
    risk_score = risk_score[mask]
 
    if len(time) < 2 or int(event.sum()) == 0:
        return np.nan
 
    order = np.argsort(-time, kind="mergesort")
    time = time[order]
    event = event[order]
    risk_score = risk_score[order]
 
    unique_scores = np.unique(risk_score)
    risk_rank = np.searchsorted(unique_scores, risk_score, side="left") + 1
    tree = _FenwickTree(len(unique_scores))
 
    concordant = 0.0
    tied_risk = 0.0
    comparable = 0
    n_later = 0
 
    i = 0
    n = len(time)
    while i < n:
        j = i + 1
        while j < n and time[j] == time[i]:
            j += 1
 
        for k in range(i, j):
            if event[k]:
                rank = int(risk_rank[k])
                n_lower_risk = tree.prefix_sum(rank - 1)
                n_equal_risk = tree.prefix_sum(rank) - n_lower_risk
 
                concordant += n_lower_risk
                tied_risk += n_equal_risk
                comparable += n_later
 
        for k in range(i, j):
            tree.add(risk_rank[k], 1)
            n_later += 1
 
        i = j
 
    if comparable == 0:
        return np.nan
 
    return (concordant + 0.5 * tied_risk) / comparable
 
 
def make_train_test_mask(base_mask, event_values, test_size=0.30, seed=42):
    """Stratified train/test split based on death event status."""
    rng = np.random.default_rng(seed)
    base_mask = np.asarray(base_mask, dtype=bool)
    event_values = np.asarray(event_values, dtype=bool)
 
    eligible_idx = np.where(base_mask)[0]
    event_idx = eligible_idx[event_values[eligible_idx]]
    censored_idx = eligible_idx[~event_values[eligible_idx]]
 
    rng.shuffle(event_idx)
    rng.shuffle(censored_idx)
 
    n_event_test = max(1, int(round(len(event_idx) * test_size)))
    n_censored_test = max(1, int(round(len(censored_idx) * test_size)))
 
    test_idx = np.concatenate([event_idx[:n_event_test], censored_idx[:n_censored_test]])
    train_idx = np.concatenate([event_idx[n_event_test:], censored_idx[n_censored_test:]])
 
    train_mask = np.zeros(len(base_mask), dtype=bool)
    test_mask = np.zeros(len(base_mask), dtype=bool)
    train_mask[train_idx] = True
    test_mask[test_idx] = True
 
    return train_mask, test_mask
 
 
def make_stratified_kfold_masks(base_mask, event_values, n_splits=5, n_repeats=5, seed=42):
    """
    Returns a list of dictionaries, where each element has the form:
{"rep": ..., "fold": ..., "train_mask": ..., "test_mask": ...}.
    `n_splits` controls the train/test ratio in each fold
(e.g., 5 folds correspond to an 80/20 split).
    `n_repeats` repeats the entire K-fold cross-validation using different
random seeds to reduce the variability associated with a single random split.
    """
    base_mask = np.asarray(base_mask, dtype=bool)
    event_values = np.asarray(event_values, dtype=bool)
    eligible_idx = np.where(base_mask)[0]
 
    splits = []
    for rep in range(n_repeats):
        rng = np.random.default_rng(seed + rep)
        event_idx = eligible_idx[event_values[eligible_idx]].copy()
        censored_idx = eligible_idx[~event_values[eligible_idx]].copy()
        rng.shuffle(event_idx)
        rng.shuffle(censored_idx)
 
        event_folds = np.array_split(event_idx, n_splits)
        censored_folds = np.array_split(censored_idx, n_splits)
 
        for fold in range(n_splits):
            test_idx = np.concatenate([event_folds[fold], censored_folds[fold]])
            train_idx = np.concatenate(
                [event_folds[i] for i in range(n_splits) if i != fold]
                + [censored_folds[i] for i in range(n_splits) if i != fold]
            )
            train_mask = np.zeros(len(base_mask), dtype=bool)
            test_mask = np.zeros(len(base_mask), dtype=bool)
            train_mask[train_idx] = True
            test_mask[test_idx] = True
            splits.append({"rep": rep, "fold": fold, "train_mask": train_mask, "test_mask": test_mask})
    return splits
 

In [12]:
# Candidate disease selection (performed on the training set only to prevent information leakage)
 
min_exposed_for_multivariable = 50
min_exposed_event_for_multivariable = 10
test_size_for_cox = 0.20
cox_split_seed = 42
 
full_model_mask_np = (
    adjusted_model_mask.to_numpy()
    & np.isfinite(baseline_feature_df.to_numpy()).all(axis=1)
)
train_mask_np, test_mask_np = make_train_test_mask(
    base_mask=full_model_mask_np,
    event_values=event.to_numpy(),
    test_size=test_size_for_cox,
    seed=cox_split_seed,
)
 
time_all = duration_days.to_numpy()
event_all = event.to_numpy()
 
print("Cox train/test split")
print("Train rows:", int(train_mask_np.sum()), "Train deaths:", int(event_all[train_mask_np].sum()))
print("Test rows:", int(test_mask_np.sum()), "Test deaths:", int(event_all[test_mask_np].sum()))
 
train_adjusted_records = []
for feature_name in baseline_feature_df.columns:
    if feature_name == "baseline_disease_count":
        continue
 
    x_disease = baseline_feature_df[feature_name].to_numpy().astype(float)
    n_exposed_train = int(x_disease[train_mask_np].sum())
    n_exposed_event_train = int(event_all[train_mask_np & (x_disease == 1)].sum())
 
    if n_exposed_train < min_exposed_for_multivariable:
        continue
    if n_exposed_event_train < min_exposed_event_for_multivariable:
        continue
 
    X_screen_df = pd.concat(
        [pd.Series(x_disease, name=feature_name, index=cluster_df.index), adjusted_covariate_df],
        axis=1,
    )
 
    screen_result_df = fit_cox_model(
        time=time_all[train_mask_np],
        event=event_all[train_mask_np],
        X=X_screen_df.to_numpy()[train_mask_np],
        feature_names=X_screen_df.columns.tolist(),
    )
    if screen_result_df is None:
        continue
 
    disease_row = screen_result_df.loc[screen_result_df["feature"] == feature_name].iloc[0].to_dict()
    disease_row["n_exposed_train"] = n_exposed_train
    disease_row["n_exposed_event_train"] = n_exposed_event_train
    disease_row["train_prevalence_percent"] = n_exposed_train / int(train_mask_np.sum()) * 100
    train_adjusted_records.append(disease_row)
 
train_age_sex_adjusted_screening_df = pd.DataFrame(train_adjusted_records)
if train_age_sex_adjusted_screening_df.empty:
    raise ValueError("No diseases passed the train-set screening criteria.")
 
train_age_sex_adjusted_screening_df = train_age_sex_adjusted_screening_df.sort_values(
    ["p_value", "hr"], ascending=[True, False]
).reset_index(drop=True)
 
candidate_diseases = train_age_sex_adjusted_screening_df["feature"].tolist()
age_sex_cols = adjusted_covariate_df.columns.tolist()
final_predictor_cols = age_sex_cols + candidate_diseases
 
print(f"\n{len(candidate_diseases)} diseases passed training-set screening "
      f"(n_exposed>={min_exposed_for_multivariable}, "
      f"n_exposed_event>={min_exposed_event_for_multivariable})")
 
screening_file = output_dir / "cox_train_screening_candidate_diseases.csv"
screening_file = safe_to_csv(train_age_sex_adjusted_screening_df, screening_file)
print(f"Saved screening table to: {screening_file}")
 

Cox train/test split
Train rows: 11525 Train deaths: 5173
Test rows: 2881 Test deaths: 1293

47 diseases passed training-set screening (n_exposed>=50, n_exposed_event>=10)
Saved screening table to: cox_vs_rsf_2012_cluster1_results/cox_train_screening_candidate_diseases.csv


In [13]:
#  Cox and RSF same k-fold 
 
X_final_df = pd.concat([adjusted_covariate_df, baseline_feature_df], axis=1)
X_final_df = X_final_df[final_predictor_cols].astype(float)
 
n_splits_final_cv = 5
n_repeats_final_cv = 5     
seed_final_cv = 42
 
final_cv_splits = make_stratified_kfold_masks(
    base_mask=full_model_mask_np,
    event_values=event_all,
    n_splits=n_splits_final_cv,
    n_repeats=n_repeats_final_cv,
    seed=seed_final_cv,
)
print(f"Total folds: {len(final_cv_splits)} ({n_splits_final_cv} splits x {n_repeats_final_cv} repeats)")
print(f"Final model uses {len(final_predictor_cols)} predictors "
      f"({len(candidate_diseases)} diseases + age/sex).")
 

Total folds: 25 (5 splits x 5 repeats)
Final model uses 49 predictors (47 diseases + age/sex).


In [14]:
#  Cox repeated k-fold CV（held-out C-index）
 
cox_final_cv_records = []
 
for split in final_cv_splits:
    fold_train_mask = split["train_mask"]
    fold_test_mask = split["test_mask"]
 
    cox_fit_df = fit_cox_model(
        time=time_all[fold_train_mask],
        event=event_all[fold_train_mask],
        X=X_final_df.to_numpy()[fold_train_mask],
        feature_names=final_predictor_cols,
    )
    if cox_fit_df is None:
        print(f"Rep {split['rep']} Fold {split['fold']}: Cox model failed to fit, skipped.")
        continue
 
    coef_map = cox_fit_df.set_index("feature")["coef"].to_dict()
    coef_vector = np.array([coef_map.get(c, 0.0) for c in final_predictor_cols], dtype=float)
 
    risk_test = X_final_df.to_numpy()[fold_test_mask] @ coef_vector
    test_c = concordance_index_censored(time_all[fold_test_mask], event_all[fold_test_mask], risk_test)
 
    cox_final_cv_records.append({"rep": split["rep"], "fold": split["fold"], "test_c_index": test_c})
 
cox_final_cv_df = pd.DataFrame(cox_final_cv_records)
 
cox_final_cv_summary = {
    "model": f"Cox (age+sex+{len(candidate_diseases)} diseases)",
    "n_folds": len(cox_final_cv_df),
    "mean_c_index": cox_final_cv_df["test_c_index"].mean(),
    "std_c_index": cox_final_cv_df["test_c_index"].std(),
    "ci_low": cox_final_cv_df["test_c_index"].quantile(0.025),
    "ci_high": cox_final_cv_df["test_c_index"].quantile(0.975),
}
 
print("\nCox repeated k-fold CV summary:")
print(cox_final_cv_summary)
 
cox_final_cv_file = output_dir / "cox_final_model_repeated_kfold_cindex.csv"
cox_final_cv_file = safe_to_csv(cox_final_cv_df, cox_final_cv_file)
 



Cox repeated k-fold CV summary:
{'model': 'Cox (age+sex+47 diseases)', 'n_folds': 25, 'mean_c_index': np.float64(0.7399522203954784), 'std_c_index': np.float64(0.004437747409655552), 'ci_low': np.float64(0.7319535508102877), 'ci_high': np.float64(0.7458658254311077)}


In [15]:
#  RSF hyperparameters
# Best hyperparameters obtained from the grid search in
# 8020cox_survival_vs_RSF_analysis_2012_cluster (Cell 33).
# They are hard-coded here for reproducibility and efficiency.
# Re-run the grid search whenever the candidate disease list,
# feature matrix, or training/test split is modified.

best_params_rsf = {"max_features": "sqrt", "min_samples_leaf": 5}
print("Using RSF hyperparameters:", best_params_rsf)
 

Using RSF hyperparameters: {'max_features': 'sqrt', 'min_samples_leaf': 5}


In [16]:
# #Repeated K-fold cross-validation for the RSF model

# Uses exactly the same train/test folds as the Cox model to enable
# paired performance comparison.
 
rsf_n_estimators = 200          
rsf_min_samples_leaf = best_params_rsf["min_samples_leaf"]
rsf_max_features = best_params_rsf["max_features"]
rsf_random_state = 42
 
rsf_final_cv_records = []
 
for split in final_cv_splits:
    fold_train_mask = split["train_mask"]
    fold_test_mask = split["test_mask"]
 
    y_train = Surv.from_arrays(event=event_all[fold_train_mask], time=time_all[fold_train_mask])
 
    rsf = RandomSurvivalForest(
        n_estimators=rsf_n_estimators,
        min_samples_leaf=rsf_min_samples_leaf,
        max_features=rsf_max_features,
        max_depth=12,          
        n_jobs=1,               
        random_state=rsf_random_state,
    )
    rsf.fit(X_final_df.to_numpy()[fold_train_mask], y_train)
 
    rsf_risk_test = rsf.predict(X_final_df.to_numpy()[fold_test_mask])
    test_c = concordance_index_censored(time_all[fold_test_mask], event_all[fold_test_mask], rsf_risk_test)
 
    rsf_final_cv_records.append({"rep": split["rep"], "fold": split["fold"], "test_c_index": test_c})
    print(f"Rep {split['rep']} Fold {split['fold']}: RSF test C-index = {test_c:.4f}")
 
    del rsf, y_train, rsf_risk_test
    gc.collect()
 
rsf_final_cv_df = pd.DataFrame(rsf_final_cv_records)
 
rsf_final_cv_summary = {
    "model": f"RSF (age+sex+{len(candidate_diseases)} diseases, tuned params)",
    "n_folds": len(rsf_final_cv_df),
    "mean_c_index": rsf_final_cv_df["test_c_index"].mean(),
    "std_c_index": rsf_final_cv_df["test_c_index"].std(),
    "ci_low": rsf_final_cv_df["test_c_index"].quantile(0.025),
    "ci_high": rsf_final_cv_df["test_c_index"].quantile(0.975),
}
 
print("\nRSF repeated k-fold CV summary:")
print(rsf_final_cv_summary)
 
rsf_final_cv_file = output_dir / "rsf_final_model_repeated_kfold_cindex.csv"
rsf_final_cv_file = safe_to_csv(rsf_final_cv_df, rsf_final_cv_file)
 

Rep 0 Fold 0: RSF test C-index = 0.7377
Rep 0 Fold 1: RSF test C-index = 0.7371
Rep 0 Fold 2: RSF test C-index = 0.7362
Rep 0 Fold 3: RSF test C-index = 0.7351
Rep 0 Fold 4: RSF test C-index = 0.7281
Rep 1 Fold 0: RSF test C-index = 0.7331
Rep 1 Fold 1: RSF test C-index = 0.7377
Rep 1 Fold 2: RSF test C-index = 0.7300
Rep 1 Fold 3: RSF test C-index = 0.7367
Rep 1 Fold 4: RSF test C-index = 0.7325
Rep 2 Fold 0: RSF test C-index = 0.7297
Rep 2 Fold 1: RSF test C-index = 0.7418
Rep 2 Fold 2: RSF test C-index = 0.7319
Rep 2 Fold 3: RSF test C-index = 0.7271
Rep 2 Fold 4: RSF test C-index = 0.7402
Rep 3 Fold 0: RSF test C-index = 0.7328
Rep 3 Fold 1: RSF test C-index = 0.7361
Rep 3 Fold 2: RSF test C-index = 0.7392
Rep 3 Fold 3: RSF test C-index = 0.7316
Rep 3 Fold 4: RSF test C-index = 0.7339
Rep 4 Fold 0: RSF test C-index = 0.7347
Rep 4 Fold 1: RSF test C-index = 0.7383
Rep 4 Fold 2: RSF test C-index = 0.7400
Rep 4 Fold 3: RSF test C-index = 0.7328
Rep 4 Fold 4: RSF test C-index = 0.7275


In [17]:
#  Cox vs RSF 
 
paired_df = cox_final_cv_df.merge(rsf_final_cv_df, on=["rep", "fold"], suffixes=("_cox", "_rsf"))
paired_df["delta_rsf_minus_cox"] = paired_df["test_c_index_rsf"] - paired_df["test_c_index_cox"]
 
delta_summary = {
    "cox_mean_c_index": cox_final_cv_summary["mean_c_index"],
    "rsf_mean_c_index": rsf_final_cv_summary["mean_c_index"],
    "mean_delta_rsf_minus_cox": paired_df["delta_rsf_minus_cox"].mean(),
    "std_delta": paired_df["delta_rsf_minus_cox"].std(),
    "delta_ci_low": paired_df["delta_rsf_minus_cox"].quantile(0.025),
    "delta_ci_high": paired_df["delta_rsf_minus_cox"].quantile(0.975),
    "pct_folds_rsf_better": float((paired_df["delta_rsf_minus_cox"] > 0).mean() * 100),
}
 
print("\nCox vs RSF paired comparison:")
print(f"Cox mean C-index:  {delta_summary['cox_mean_c_index']:.4f}")
print(f"RSF mean C-index:  {delta_summary['rsf_mean_c_index']:.4f}")
print(f"Paired delta (RSF - Cox): mean={delta_summary['mean_delta_rsf_minus_cox']:.4f}, "
      f"95% CI=[{delta_summary['delta_ci_low']:.4f}, {delta_summary['delta_ci_high']:.4f}]")
print(f"RSF wins in {delta_summary['pct_folds_rsf_better']:.1f}% of folds")
 
if delta_summary["delta_ci_low"] > 0:
    print("=> RSF significantly outperformed the Cox model (95% CI entirely above 0).")
elif delta_summary["delta_ci_high"] < 0:
    print("=> The Cox model significantly outperformed RSF (95% CI entirely below 0).")
else:
    print("=> No statistically significant difference was observed between the two models (95% CI crosses 0), suggesting that the linear additive assumption provides a reasonable approximation for this population.")
    
paired_file = output_dir / "cox_vs_rsf_paired_delta_cindex.csv"
paired_file = safe_to_csv(paired_df, paired_file)
print(f"\nSaved paired comparison to: {paired_file}")
 


Cox vs RSF paired comparison:
Cox mean C-index:  0.7400
RSF mean C-index:  0.7345
Paired delta (RSF - Cox): mean=-0.0055, 95% CI=[-0.0097, -0.0001]
RSF wins in 4.0% of folds
=> Cox 显著优于 RSF（95% CI 完全在0以下）

Saved paired comparison to: cox_vs_rsf_2012_cluster1_results/cox_vs_rsf_paired_delta_cindex.csv
